In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


In [2]:
airline_data = pd.read_csv('data/airline_passenger_satisfaction.csv')
airline_data.sample(10)

,Unnamed: 0,ID,Gender,Age,Customer Type,Type of Travel,Class,Flight Distance,Departure Delay,Arrival Delay,...,On-board Service,Seat Comfort,Leg Room Service,Cleanliness,Food and Drink,In-flight Service,In-flight Wifi Service,In-flight Entertainment,Baggage Handling,Satisfaction
93434,93434,93435,FEMALE,65,Returning,Business,Business,3822 miles,0 minutes,0.0 minutes,...,3,3,3,1,1,3,3,3,3,Neutral or Dissatisfied
40514,40514,40515,Female,40,Returning,Business,Economy Plus,222 miles,0 minutes,0.0 minutes,...,4,4,2,4,4,5,4,4,2,Satisfied
88737,88737,88738,male,52,Returning,Personal,ECONOMY,444 miles,0 minutes,0.0 minutes,...,4,2,2,2,2,5,2,2,5,Neutral or Dissatisfied
64098,64098,64099,Female,24,Returning,Personal,Economy,919 miles,0 minutes,6.0 minutes,...,4,2,5,2,2,4,1,2,3,Neutral or Dissatisfied
8499,8499,8500,Male,48,returning,Business,Business,3522 miles,34 minutes,9.0 minutes,...,5,5,5,4,2,5,2,5,5,Satisfied
79468,79468,79469,Male,68,returning,PERSONAL,ECONOMY,1183 miles,37 minutes,80.0 minutes,...,5,3,4,3,3,5,1,3,4,Neutral or Dissatisfied
15231,15231,15232,Male,43,Returning,Business,Business,416 miles,0 minutes,10.0 minutes,...,5,1,5,2,1,5,4,5,5,Satisfied
81077,81077,81078,Female,45,Returning,Business,Business,931 miles,3 minutes,0.0 minutes,...,2,5,2,4,5,2,2,2,2,Satisfied
100878,100878,100879,male,41,First-time,business,ECONOMY,739 miles,15 minutes,7.0 minutes,...,1,1,2,2,2,2,4,2,3,Neutral or Dissatisfied
68628,68628,68629,Female,39,Returning,Business,Business,3978 miles,0 minutes,0.0 minutes,...,3,4,4,5,4,3,5,3,3,Satisfied


In [3]:
# Check the value counts for categorical columns and some numerical columns
cols_to_check = ['Gender', 'Customer Type', 'Type of Travel', 'Class',
                 'Flight Distance', 'Departure Delay', 'Arrival Delay']

for col in cols_to_check:
    print(f"--- {col} ---") #prints the column name as a header for the value counts
    print(airline_data[col].value_counts(dropna=False).head(6).to_string())
    print()


--- Gender ---
Gender
Female    48470
Male      46992
FEMALE    20773
male      20139

--- Customer Type ---
Customer Type
Returning       77949
returning       33407
First-time      17513
 First-time      7505

--- Type of Travel ---
Type of Travel
Business    65875
Personal    29587
business    28232
PERSONAL    12680

--- Class ---
Class
Business        45674
Economy         42893
 Business       19574
ECONOMY         18382
Economy Plus     6896
ECONOMY PLUS     2955

--- Flight Distance ---
Flight Distance
337 miles     881
594 miles     530
404 miles     501
862 miles     499
2475 miles    496
447 miles     476

--- Departure Delay ---
Departure Delay
0 minutes    77017
1 minutes     3862
2 minutes     3009
3 minutes     2666
4 minutes     2409
5 minutes     2249

--- Arrival Delay ---
Arrival Delay
0.0 minutes    76402
1.0 minutes     2877
2.0 minutes     2719
3.0 minutes     2553
4.0 minutes     2491
5.0 minutes     2197



In [4]:
#Drop the 'Unnamed' column.
airline_data = airline_data.drop(columns=['Unnamed: 0'])
print("Dropped 'Unnamed'")
print("Remaining columns:", airline_data.columns.tolist())


Dropped 'Unnamed'
Remaining columns: ['ID', 'Gender', 'Age', 'Customer Type', 'Type of Travel', 'Class', 'Flight Distance', 'Departure Delay', 'Arrival Delay', 'Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service', 'Online Boarding', 'Gate Location', 'On-board Service', 'Seat Comfort', 'Leg Room Service', 'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service', 'In-flight Entertainment', 'Baggage Handling', 'Satisfaction']


In [5]:
#fix cateogorical columns by stripping whitespace and standardizing capitalization
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

for col in cat_cols:
    airline_data[col] = airline_data[col].str.strip().str.title()

for col in cat_cols:
    print(f"--- {col} ---")
    print(airline_data[col].value_counts().to_string())
    print()


--- Gender ---
Gender
Female    69243
Male      67131

--- Customer Type ---
Customer Type
Returning     111356
First-Time     25018

--- Type of Travel ---
Type of Travel
Business    94107
Personal    42267

--- Class ---
Class
Business        65248
Economy         61275
Economy Plus     9851



In [6]:
#fix 'Flight Distance' by removing ' miles', stripping whitespace, and converting to int

airline_data['Flight Distance'] = (
    airline_data['Flight Distance']
    .str.replace(' miles', '', regex=False)
    .str.strip()
    .astype(int)
)


In [7]:
print("======Flight Distance======")
print(f"dtype: {airline_data['Flight Distance'].dtype}")
print(f"Range: {airline_data['Flight Distance'].min()} – {airline_data['Flight Distance'].max()} miles")
airline_data['Flight Distance'].describe()

======Flight Distance======
dtype: int64
Range: 31 – 4983 miles


count    136374.000000
mean       1188.614604
std         996.663096
min          31.000000
25%         413.000000
50%         843.000000
75%        1741.000000
max        4983.000000
Name: Flight Distance, dtype: float64

In [8]:
airline_data['Departure Delay'] = (
    airline_data['Departure Delay']
    .str.replace(' minutes', '', regex=False)
    .str.strip()
    .astype(int)
)

In [9]:
print("======Departure Delay======")
print(f"dtype: {airline_data['Departure Delay'].dtype}")
print(f"Range: {airline_data['Departure Delay'].min()} – {airline_data['Departure Delay'].max()} minutes")
airline_data['Departure Delay'].describe()


======Departure Delay======
dtype: int64
Range: 0 – 1592 minutes


count    136374.000000
mean         14.727243
std          38.221892
min           0.000000
25%           0.000000
50%           0.000000
75%          12.000000
max        1592.000000
Name: Departure Delay, dtype: float64

In [ ]:
# Convert Arrival Delay to numeric, then fill missing values with the Departure Delay of the same row
airline_data['Arrival Delay'] = (
    airline_data['Arrival Delay']
    .str.replace(' minutes', '', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

airline_data['Arrival Delay'] = airline_data['Arrival Delay'].fillna(airline_data['Departure Delay']).astype(int)

# Confirm no missing values remain
print(f"Missing values remaining: {airline_data['Arrival Delay'].isna().sum()}") 

Missing values remaining: 0


In [11]:
print("======Departure Delay======")
print(f"dtype: {airline_data['Departure Delay'].dtype}")
print(f"Remaining nulls: {airline_data['Departure Delay'].isna().sum()}")
print(f"Range: {airline_data['Departure Delay'].min()} – {airline_data['Departure Delay'].max()} minutes")
airline_data['Departure Delay'].describe()


======Departure Delay======
dtype: int64
Remaining nulls: 0
Range: 0 – 1592 minutes


count    136374.000000
mean         14.727243
std          38.221892
min           0.000000
25%           0.000000
50%           0.000000
75%          12.000000
max        1592.000000
Name: Departure Delay, dtype: float64

In [13]:
airline_data.to_csv("data/airline_passenger_satisfaction_cleaned.csv", index=False)
print(f"Cleaned dataset saved to: airline_passenger_satisfaction_cleaned.csv")
print(f"Final shape: {airline_data.shape}")


Cleaned dataset saved to: airline_passenger_satisfaction_cleaned.csv
Final shape: (136374, 24)
